In [ ]:
%pip install --upgrade pip
%pip install llama-index-utils-workflow
%pip install datasets
%pip install llama-index-retrievers-bm25
%pip install -q --upgrade llama-index-llms-openai
%pip install llama-index-tools-duckduckgo
%pip install ddgs huggingface_hub


### Creating a RAG Tool for Guest Stories


In [19]:
import datasets
from llama_index.core.schema import Document

# Load the dataset
guest_dataset = datasets.load_dataset("agents-course/unit3-invitees", split="train")

# Convert dataset entries into Document objects
docs = [
    Document(
        text="\n".join([
            f"Name: {guest_dataset['name'][i]}",
            f"Relation: {guest_dataset['relation'][i]}",
            f"Description: {guest_dataset['description'][i]}",
            f"Email: {guest_dataset['email'][i]}"
        ]),
        metadata={"name": guest_dataset['name'][i]}
    )
    for i in range(len(guest_dataset))
]


In [20]:
from llama_index.core.tools import FunctionTool
from llama_index.retrievers.bm25 import BM25Retriever

bm25_retriever = BM25Retriever.from_defaults(nodes=docs)

def get_guest_info_retriever(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = bm25_retriever.retrieve(query)
    if results:
        return "\n\n".join([doc.text for doc in results[:3]])
    else:
        return "No matching guest information found."

# Initialize the tool
guest_info_tool = FunctionTool.from_defaults(get_guest_info_retriever)


In [21]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.openai import OpenAI
from getpass import getpass
import os

# Initialize the OpenAI model
openai_api_key = os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")
llm = OpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
)

# Create Alfred, our gala agent, with the guest info tool
alfred = AgentWorkflow.from_tools_or_functions(
    [guest_info_tool],
    llm=llm,
)

# Example query Alfred might receive during the gala
response = await alfred.run(
    user_msg="Tell me about our guest named 'Lady Ada Lovelace'."
)

print("🎩 Alfred's Response:")
print(response)


🎩 Alfred's Response:
Lady Ada Lovelace is my best friend and an esteemed mathematician. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her contributions to Charles Babbage's Analytical Engine. 

You can reach her at her email: ada.lovelace@example.com.


### Building and Integrating Tools for Your Agent


In [26]:
from llama_index.tools.duckduckgo import DuckDuckGoSearchToolSpec
import time

# Initialize the DuckDuckGo search tool
tool_spec = DuckDuckGoSearchToolSpec()

search_tool = FunctionTool.from_defaults(tool_spec.duckduckgo_full_search)
query = "Who is the current President of France?"
result_text = None

# Example usage
for attempt in range(5):
    try:
        search_response = search_tool(
            query=query,
            max_results=5,
        )
        if search_response.raw_output:
            result_text = search_response.raw_output[-1]["body"]
        break
    except Exception as exc:
        if "ratelimit" not in str(exc).lower():
            raise
        if attempt < 4:
            time.sleep(2 ** (attempt + 1))

# Fall back to the existing LLM when DuckDuckGo remains rate-limited.
if result_text is None:
    result_text = str(llm.complete(f"Answer this question concisely: {query}"))

print(result_text)


As of October 2023, the current President of France is Emmanuel Macron.


In [27]:
import random
from llama_index.core.tools import FunctionTool

def get_weather_info(location: str) -> str:
    """Fetches dummy weather information for a given location."""
    # Dummy weather data
    weather_conditions = [
        {"condition": "Rainy", "temp_c": 15},
        {"condition": "Clear", "temp_c": 25},
        {"condition": "Windy", "temp_c": 20}
    ]
    # Randomly select a weather condition
    data = random.choice(weather_conditions)
    return f"Weather in {location}: {data['condition']}, {data['temp_c']}°C"

# Initialize the tool
weather_info_tool = FunctionTool.from_defaults(get_weather_info)


In [ ]:
from llama_index.core.tools import FunctionTool
from huggingface_hub import list_models

def get_hub_stats(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        # List models from the specified author, sorted by downloads
        models = list(list_models(author=author, sort="downloads", direction=-1, limit=1))

        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        else:
            return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

# Initialize the tool
hub_stats_tool = FunctionTool.from_defaults(get_hub_stats)

# Example usage
print(hub_stats_tool("facebook")) # Example: Get the most downloaded model by Facebook


Error fetching models for facebook: HfApi.list_models() got an unexpected keyword argument 'direction'


In [31]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.openai import OpenAI

# Initialize the OpenAI model
llm = OpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
)

# Create Alfred with all the tools
alfred = AgentWorkflow.from_tools_or_functions(
    [search_tool, weather_info_tool, hub_stats_tool],
    llm=llm,
)

# Example query Alfred might receive during the gala
response = await alfred.run(
    user_msg="What is Facebook and what's their most popular model?"
)

print("🎩 Alfred's Response:")
print(response)


🎩 Alfred's Response:
### What is Facebook?
Facebook is a social media platform that allows users to connect with friends and family, share content, and engage with communities. Founded in 2004 by Mark Zuckerberg and his college roommates, it has grown to become one of the largest social networking sites in the world. Users can create profiles, post updates, share photos and videos, and interact with others through comments, likes, and messages. Facebook also offers various features such as groups, events, and marketplace for buying and selling items.

For more detailed information, you can check the [DuckDuckGo search results](https://links.duckduckgo.com/d.js?q=What+is+Facebook%3F&kl=wt-wt&l=wt-wt&p=&s=0&df=&vqd=4-111557617445658095477167995024768950828&bing_market=wt-WT&ex=-1).

### Most Popular Model from Facebook
It seems there was an error while trying to fetch the most popular model from Facebook on the Hugging Face Hub. If you have a specific model in mind or need information ab

### Integrating Tools with Alfred


In [32]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.openai import OpenAI

# Initialize the OpenAI model
llm = OpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
)

# Create Alfred with all the tools
alfred = AgentWorkflow.from_tools_or_functions(
    [search_tool, weather_info_tool, hub_stats_tool],
    llm=llm,
)

# Example query Alfred might receive during the gala
response = await alfred.run(
    user_msg="What is Facebook and what's their most popular model?"
)

print("🎩 Alfred's Response:")
print(response)


🎩 Alfred's Response:
### What is Facebook?
Facebook is a social media platform that allows users to connect with friends, family, and others around the world. It was founded by Mark Zuckerberg and his college roommates in 2004. Users can create profiles, share updates, photos, and videos, and engage with content through likes, comments, and shares. Facebook also offers features like groups, events, and marketplace for buying and selling items.

For more detailed information, you can check the following link: [What is Facebook?](https://links.duckduckgo.com/d.js?q=What+is+Facebook%3F&kl=wt-wt&l=wt-wt&p=&s=0&df=&vqd=4-111557617445658095477167995024768950828&bing_market=wt-WT&ex=-1).

### Most Popular Model
I encountered an error while trying to fetch the most popular model from Facebook on the Hugging Face Hub. However, Facebook is known for several popular models, including:

- **BART**: A model for natural language processing tasks.
- **RoBERTa**: An optimized version of BERT for bette

### The Complete Agent


In [33]:
import os
from getpass import getpass
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.openai import OpenAI

# Initialize the OpenAI model and combine all notebook tools
openai_api_key = os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")
llm = OpenAI(
    model="gpt-4o-mini",
    api_key=openai_api_key,
)

alfred = AgentWorkflow.from_tools_or_functions(
    [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool],
    llm=llm,
)


In [34]:
### Example 1: Finding Guest Information

query = "Tell me about Lady Ada Lovelace. What's her background?"
response = await alfred.run(query)

print("🎩 Alfred's Response:")
print(response.response.blocks[0].text)


🎩 Alfred's Response:
Lady Ada Lovelace, born Augusta Ada Byron on December 10, 1815, is often regarded as the first computer programmer. She was the only legitimate child of the famous poet Lord Byron and his wife, Annabella Milbanke. Her parents separated when she was just a month old, and her father left the family, never to return. Ada's mother, who had a strong interest in mathematics, encouraged her daughter's education in this field, hoping to steer her away from the artistic tendencies of her father.

Ada showed a remarkable aptitude for mathematics and science from a young age. She studied under notable mathematicians, including Augustus De Morgan and Mary Somerville, who introduced her to the work of Charles Babbage, the inventor of the Analytical Engine, an early mechanical general-purpose computer.

In the mid-1800s, Lovelace collaborated with Babbage on his Analytical Engine. She translated an article written by the Italian mathematician Luigi Federico Federico Federico, ad

In [35]:
### Example 2: Checking the Weather for Fireworks

query = "What's the weather like in Paris tonight? Will it be suitable for our fireworks display?"
response = await alfred.run(query)

print("🎩 Alfred's Response:")
print(response)


🎩 Alfred's Response:
The weather in Paris tonight is clear with a temperature of 25°C. It seems suitable for your fireworks display!


In [36]:
### Example 3: Impressing AI Researchers

query = "One of our guests is from Google. What can you tell me about their most popular model?"
response = await alfred.run(query)

print("🎩 Alfred's Response:")
print(response)


🎩 Alfred's Response:
It seems that I'm currently unable to retrieve information about the most popular model from Google or its related entities. However, Google is known for several prominent models, including:

1. **BERT (Bidirectional Encoder Representations from Transformers)** - A transformer-based model designed for natural language processing tasks.
2. **T5 (Text-to-Text Transfer Transformer)** - A model that converts all NLP tasks into a text-to-text format.
3. **BigGAN** - A generative adversarial network for generating high-quality images.

If you have a specific model in mind or need information on a particular aspect, please let me know!


In [37]:
### Example 4: Combining Multiple Tools

query = "I need to speak with Dr. Nikola Tesla about recent advancements in wireless energy. Can you help me prepare for this conversation?"
response = await alfred.run(query)

print("🎩 Alfred's Response:")
print(response)


🎩 Alfred's Response:
To prepare for your conversation with Dr. Nikola Tesla, here are some key points and tips:

### Background Information
- **Name**: Dr. Nikola Tesla
- **Relation**: Old friend from university days
- **Recent Work**: He has recently patented a new wireless energy transmission system.
- **Personal Interest**: He is passionate about pigeons, which could be a good topic for small talk.

### Conversation Preparation
1. **Recent Advancements in Wireless Energy**:
   - Familiarize yourself with the latest trends and technologies in wireless energy transmission. This could include developments in resonant inductive coupling, microwave energy transmission, and advancements in energy efficiency.

2. **Discuss His Patent**:
   - Ask him about the specifics of his new wireless energy transmission system. What makes it unique? What are its potential applications?

3. **Small Talk**:
   - Since Dr. Tesla loves pigeons, you might want to share any interesting stories or facts abou

In [38]:
### Advanced Features: Conversation Memory

from llama_index.core.workflow import Context

alfred = AgentWorkflow.from_tools_or_functions(
    [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool],
    llm=llm
)

# Remembering state
ctx = Context(alfred)

# First interaction
response1 = await alfred.run("Tell me about Lady Ada Lovelace.", ctx=ctx)
print("🎩 Alfred's First Response:")
print(response1)

# Second interaction (referencing the first)
response2 = await alfred.run("What projects is she currently working on?", ctx=ctx)
print("🎩 Alfred's Second Response:")
print(response2)


🎩 Alfred's First Response:
Lady Ada Lovelace, born on December 10, 1815, is often regarded as the first computer programmer. She was the daughter of the famous poet Lord Byron and mathematician Annabella Milbanke. Ada showed an early interest in mathematics and science, which was encouraged by her mother.

Lovelace is best known for her work on Charles Babbage's early mechanical general-purpose computer, the Analytical Engine. She recognized that the machine had applications beyond mere calculation and wrote what is considered the first algorithm intended for implementation on a computer. This foresight into the potential of computers was groundbreaking and laid the foundation for future developments in computing.

In addition to her work on the Analytical Engine, Lovelace wrote notes that included ideas about the manipulation of symbols and the concept of programming. Her contributions were largely unrecognized during her lifetime, but she has since become a symbol of women in technol